In [7]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path
from solver import solve_pdpcr

import pickle
import copy
import datetime
from docplex.mp.model import Model
import logging
import heapq
import networkx as nx
import numpy as np
import time

logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

In [6]:

total_robots = 10
random_data = False

data_type = 'csv'
# data_name = 'generated18.csv'
# data_name = 'data3.csv'
data_name = 'lim/100/lrc203-5-uniform.csv'
# data_name = 'lim/200/LC1_2_3-5-uniform.csv'
total_tasks = 6 # random data

raw = get_data(data_name, data_type)
logging.info(f'Solving data {data_name} with {total_robots} robots and {len(raw.tasks)} tasks')
processed_h = heuristic.process_greedy_1(raw)

depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)

In [9]:
greedy_v2 = heuristic.GreedyInsert(processed_h, c_travel, c_operate)
greddy_v2_F, greedy_v2_C_max = greedy_v2.solve(total_robots)
print(f'greedy insert makespan: {greedy_v2_C_max}')

greedy insert makespan: 920.7543117467869


In [ ]:
G2 = pickle.load(open('greddy_v2_F.p', 'rb'))

In [ ]:
#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [10]:
#Li smaller data
small_first_names =['lc','lr','lrc']
small_sec_names = [100+i for i in range(1,10)] + [200 + i for i in range(1,10)]
data_paths = [f'lim/100/{i}{j}-5-uniform.csv' for i in small_first_names for j in small_sec_names]

In [11]:
num_robots = [10]
file_type = 'csv'
is_depot_task = False
#new greedy search
for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            tik1 = time.time()
            greedy_v2 = heuristic.GreedyInsert(processed_h, c_travel, c_operate)
            greedy_v2_F, greedy_v2_C_max = greedy_v2.solve(num_robot)
            tok1 = time.time()
            greedy_time = tok1 - tik1
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
                pickle.dump(greedy_v2_F, open(f'solve_results/greedy2/{data_name}.p', 'wb'))
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(greedy_v2_C_max)
            solve_log.append(np.round(greedy_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_greedy2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)
            raise(e)

25-Aug-24 03:04:02:INFO:Solving data lim/100/lc101-5-uniform.csv with 10 robots and 53 tasks



 ______________Error in lim/100/lc101-5-uniform.csv with 10 robots______________

name 'total_robots' is not defined


NameError: name 'total_robots' is not defined